# **PAAD Tutorial 1: Setting Up and Training SIDISH**

This tutorial initialises and trains the SIDISH framework on the PAAD (pancreatic adenocarcinoma) Xenium dataset, reproducing Fig 5 of the SIDISH paper.

### In this tutorial, you will:
- Load the processed single-cell and bulk RNA-seq data from Tutorial 0.
- Optionally subsample the single-cell data to reduce training time.
- Initialise SIDISH with PAAD-specific hyperparameters.
- Train the model to identify high-risk cell subpopulations.

### Paper targets (Fig 5):
| Metric | Target |
|--------|-------|
| High-risk cells | 41,323 (21.6 % of 190,965) |
| Tumour fraction in high-risk | ~67.5 % |
| Training call | `sdh.train(5, 0.95, 30, OUTPUT_DIR)` |

## **Step 1: Imports**

In [41]:
from SIDISH import SIDISH as sidish
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import random
import os
import matplotlib.pyplot as plt

In [42]:
SEED = 0
ITE  = 0  # SIDISH seed index

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## **Step 2: Configuration**

Set all run-level flags here. Flip `SUBSAMPLE` to `True` for a fast development run  
(~3 min/iteration instead of ~58 min), or keep `False` for paper-exact results.

In [43]:
# --- Run-level configuration ---
SUBSAMPLE  = True      # True → stratified 10k-cell dev run; False → full 190k paper run
N_CELLS    = 1_000     # target cell count when SUBSAMPLE=True

DATA_DIR   = "outputs/run_default"   # processed outputs from Tutorial 0
OUTPUT_DIR = "outputs/run_default"   # training artefacts written here
DEVICE     = "mps"                   # Mac — no CUDA

os.makedirs(OUTPUT_DIR, exist_ok=True)

## **Step 3: Load Processed Data**

Load the outputs saved by Tutorial 0.

### **3.1 Single-cell data**

In [44]:
adata = sc.read_h5ad(os.path.join(DATA_DIR, "processed_adata.h5ad"))
print(f"Loaded adata: {adata.n_obs:,} cells x {adata.n_vars} genes")
print(adata.obs["celltype_major"].value_counts())

Loaded adata: 190,273 cells x 474 genes
celltype_major
Macrophages                    28263
Fibroblasts                    27393
Metaplastic Cells              19476
Tumor Cells                    18837
Acinar                         18667
CFTR- Tumor Cells              18558
T Cells                        12649
Endothelial                    10315
Endocrine 1                     9223
Endocrine 2                     9163
Ductal                          4547
CXCL9/10 Cells                  3439
B Cells                         3215
Mast Cells                      2954
Smooth Muscle Cells             1897
Lymphatic Endothelial Cells     1677
Name: count, dtype: int64


### **3.2 Bulk RNA-seq + survival data**

In [45]:
bulk = pd.read_csv(os.path.join(DATA_DIR, "processed_bulk.csv"), index_col=0)
print(f"Loaded bulk: {bulk.shape[0]} patients x {bulk.shape[1]} columns")
print(f"  duration, event + {bulk.shape[1] - 2} genes")

Loaded bulk: 174 patients x 476 columns
  duration, event + 474 genes


## **Step 4: Optional Subsampling**

Training SIDISH on 190,965 cells takes ~58 minutes per iteration (SHAP is the bottleneck).  
A stratified subsample by cell type preserves biological proportions while cutting runtime to  
~3 min/iteration — useful for parameter tuning before a full paper-replication run.

Set `SUBSAMPLE = True` in Step 2 to enable. The subsampled AnnData is saved separately  
so the full dataset is never overwritten.

In [46]:
if SUBSAMPLE:
    cell_type_col = "celltype_major"
    groups  = adata.obs[cell_type_col]
    counts  = groups.value_counts()
    frac    = N_CELLS / len(adata)

    # Proportional counts per cell type; at least 1 per group
    n_per_group = (counts * frac).round().clip(lower=1).astype(int)

    # Adjust to hit exactly N_CELLS
    diff = N_CELLS - n_per_group.sum()
    if diff != 0:
        n_per_group[n_per_group.idxmax()] += diff

    rng = np.random.default_rng(SEED)
    idx = []
    for ct, n in n_per_group.items():
        pool   = adata.obs_names[groups == ct].tolist()
        chosen = rng.choice(pool, size=min(n, len(pool)), replace=False)
        idx.extend(chosen)

    adata_train = adata[idx].copy()
    sub_path    = os.path.join(DATA_DIR, f"processed_adata_{N_CELLS // 1000}k.h5ad")
    adata_train.write_h5ad(sub_path)
    print(f"Subsampled: {adata_train.n_obs:,} cells saved to {sub_path}")
    print("Cell-type distribution after subsampling:")
    print(adata_train.obs[cell_type_col].value_counts())
else:
    adata_train = adata
    print(f"Using full dataset: {adata_train.n_obs:,} cells")

Subsampled: 1,000 cells saved to outputs/run_default/processed_adata_1k.h5ad
Cell-type distribution after subsampling:
celltype_major
Macrophages                    149
Fibroblasts                    144
Metaplastic Cells              102
Tumor Cells                     99
Acinar                          98
CFTR- Tumor Cells               98
T Cells                         66
Endothelial                     54
Endocrine 1                     48
Endocrine 2                     48
Ductal                          24
CXCL9/10 Cells                  18
B Cells                         17
Mast Cells                      16
Smooth Muscle Cells             10
Lymphatic Endothelial Cells      9
Name: count, dtype: int64


## **Step 5: Initialise SIDISH**

Initialise the SIDISH model with the (possibly subsampled) single-cell data and the bulk RNA-seq data.

In [47]:
sdh = sidish(adata_train, bulk, DEVICE, seed=ITE, use_spatial_graph=False)

SIDISH No spatial graph used. Proceeding with dense VAE.


### **5.1 Initialise Phase 1 — VAE**

Phase 1 trains a Variational Autoencoder (VAE) that compresses single-cell data into a  
biologically meaningful latent space. The encoder is later reused by Phase 2.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `epoch` | 225 | VAE epochs at iteration 1 |
| `i_epoch` | 20 | VAE retraining epochs for iterations 2–5 |
| `latent_size` | 32 | Latent space dimension |
| `layer_dims` | [512, 128] | Encoder/decoder hidden layer sizes |
| `batch_size` | 512 | Cells per training batch |
| `optimizer` | Adam | Optimiser |
| `lr` | 1e-4 | Learning rate at iteration 1 |
| `lr_3` | 1e-4 | Learning rate at iterations 2–5 |
| `dropout` | 0 | Dropout rate |

In [48]:
sdh.init_Phase1(225, 20, 32, [512, 128], 512, "Adam", 1.0e-4, 1e-4, 0)

### **5.2 Initialise Phase 2 — Deep Cox Regression**

Phase 2 attaches a Deep Cox regression head to the VAE encoder and trains it on bulk  
RNA-seq survival data via transfer learning.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `epoch` | 500 | Cox training epochs |
| `hidden` | 128 | Additional FC layer size |
| `lr` | 1e-4 | Learning rate |
| `dropout` | 0 | Dropout rate |
| `test_size` | 0.2 | Fraction held out for validation |
| `batch_size` | 256 | Patients per batch |

In [49]:
sdh.init_Phase2(500, 128, 1e-4, 0, 0.2, 256)

## **Step 6: Train SIDISH**

`sdh.train()` runs the iterative SIDISH loop:
1. Train VAE on (weighted) single-cell data.
2. Train Deep Cox on bulk survival data (transfer learning from VAE encoder).
3. Score every cell; apply Weibull-fitted threshold to label high-risk cells.
4. Compute SHAP-based gene weights → update weight matrix.
5. Repeat from step 1 with the new weights.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `iterations` | 5 | Number of SIDISH iterations |
| `percentile` | 0.90 | Risk score threshold (higher → fewer high-risk cells) |
| `steepness` | 30 | Sigmoid steepness for gene weight generation |

> **Note:** Do NOT pass `distribution_fit='fitted'` — it selects a suboptimal distribution  
> that produces an extremely high threshold, causing most high-risk cells to be missed.
>
> **Runtime estimate:** ~58 min/iteration for 190k cells; ~3 min/iteration for 10k subsample.

In [50]:
train_adata = sdh.train(5, 0.90, 30, OUTPUT_DIR)

########################################## Using Dense VAE ##########################################
########################################## ITERATION 1 OUT OF 5 ##########################################
[epoch 000]  average training loss: nan
[epoch 001]  average training loss: nan
[epoch 002]  average training loss: nan
[epoch 003]  average training loss: nan
[epoch 004]  average training loss: nan
[epoch 005]  average training loss: nan
[epoch 006]  average training loss: nan
[epoch 007]  average training loss: nan
[epoch 008]  average training loss: nan
[epoch 009]  average training loss: nan
[epoch 010]  average training loss: nan
[epoch 011]  average training loss: nan
[epoch 012]  average training loss: nan
[epoch 013]  average training loss: nan
[epoch 014]  average training loss: nan
[epoch 015]  average training loss: nan
[epoch 016]  average training loss: nan
[epoch 017]  average training loss: nan
[epoch 018]  average training loss: nan
[epoch 019]  average training l

100%|██████████| 500/500 [00:02<00:00, 224.11it/s]


ValueError: NaNs detected in inputs, please correct or drop.